# Phase 3 Step 1b consumer -- stitch + verify the corpus labels

CPU-only. Attaches both shard outputs plus the bake-off output, and only produces the Phase 4
label file if every gate passes:

1. **Completion sentinels** present for both shards.
2. **Exact partition**: every one of the 4,407 train UIDs appears exactly once across the two
   shards, none missing, none unexpected (same check that validated the Phase 2 prep shards).
3. **Score sanity**: no NaN anywhere, scores and weights inside [0, 1].
4. **Gold cross-check**: the 58 gold studies were labeled here AND in the bake-off with the same
   model and prompts -- their score matrices must agree to within batching numerics. This catches
   any prompt-construction drift between kernels; it is why the bake-off output is attached.
5. Recomputed gold macro AUC from the stitched file matches the recorded 0.8613.

Writes `pseudo_labels_qwen3_4b.csv` (the Phase 4 label source) plus a verification manifest.


In [ ]:
import glob, json, os
import numpy as np
import pandas as pd

COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
BAKEOFF_DIR = glob.glob('/kaggle/input/notebooks/*/knee-phase3-bakeoff')[0]
shard_files = sorted(glob.glob('/kaggle/input/**/scores_shard*.csv', recursive=True))
sentinels = sorted(glob.glob('/kaggle/input/**/_SHARD_*_COMPLETE', recursive=True))
print('shard score files:', [f.rsplit('/', 1)[-1] for f in shard_files])
print('sentinels:', [os.path.basename(s) for s in sentinels])
assert len(shard_files) == 2, f'expected 2 shard score files, found {len(shard_files)}'
found_sentinels = {os.path.basename(s) for s in sentinels}
for i in (0, 1):
    assert f'_SHARD_{i}_COMPLETE' in found_sentinels, (
        f'shard {i} has no completion sentinel -- it did not finish; re-run that shard')


In [ ]:
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
SCORE_COLS = [f'score_{l}' for l in LABELS]
WEIGHT_COLS = [f'weight_{l}' for l in LABELS]

frames = [pd.read_csv(f) for f in shard_files]
stitched = pd.concat(frames, ignore_index=True)

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = set(train_df['StudyInstanceUID'].astype(str))
stitched_uids = stitched['StudyInstanceUID'].astype(str)
dup = stitched_uids[stitched_uids.duplicated()].unique().tolist()
missing = sorted(all_uids - set(stitched_uids))
unexpected = sorted(set(stitched_uids) - all_uids)
assert not dup, f'duplicated UIDs across shards: {dup[:5]}...'
assert not missing, f'{len(missing)} train UIDs missing from shards, e.g. {missing[:3]}'
assert not unexpected, f'unexpected UIDs in shards: {unexpected[:3]}'
stitched = stitched.set_index('StudyInstanceUID').loc[sorted(all_uids)].reset_index()

vals = stitched[SCORE_COLS + WEIGHT_COLS].to_numpy(dtype=float)
assert not np.isnan(vals).any(), 'NaN found in stitched scores/weights'
assert (vals[:, :12] >= 0).all() and (vals[:, :12] <= 1).all(), 'score outside [0,1]'
assert (vals[:, 12:] >= 0).all() and (vals[:, 12:] <= 1).all(), 'weight outside [0,1]'
print(f'partition + sanity PASS: {len(stitched)} studies x 12 labels, all finite and in range')


In [ ]:
from sklearn.metrics import roc_auc_score

gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58
gold_uids = gold_df['StudyInstanceUID'].astype(str).tolist()

bake_scores = np.load(f'{BAKEOFF_DIR}/scores_Qwen3-4B-Instruct.npy')  # (58, 12), gold_df order
here = stitched.set_index('StudyInstanceUID').loc[gold_uids][SCORE_COLS].to_numpy(dtype=float)
delta = np.abs(here - bake_scores)
print(f'gold cross-check vs bake-off: mean|d|={delta.mean():.5f} '
      f'p99={np.percentile(delta, 99):.4f} max={delta.max():.4f}')
# Two gates, deliberately different: the MEAN catches systematic drift (a prompt
# or decoding difference moves every score); the MAX is reported, not asserted,
# because fp16 batching numerics legitimately flip a few knife-edge prompts
# (score~0.5) by large amounts without changing any conclusion.
assert delta.mean() < 0.02, (
    'corpus-pass scores diverge from the bake-off on average -- prompt '
    'construction or decoding differs between kernels; investigate before trusting')

y_true = gold_df[LABELS].to_numpy(dtype=float)
aucs = []
for i in range(12):
    mask = ~np.isnan(y_true[:, i])
    aucs.append(roc_auc_score(y_true[mask, i], here[mask, i]))
macro_here = float(np.mean(aucs))
print(f'recomputed gold macro AUC from stitched file: {macro_here:.4f} (recorded: 0.8613)')
assert abs(macro_here - 0.8613) < 0.01, 'recomputed macro AUC drifted from the recorded value'


In [ ]:
OUT = 'pseudo_labels_qwen3_4b.csv'
stitched.to_csv(OUT, index=False)

verification = {
    'n_studies': int(len(stitched)),
    'n_labels': 12,
    'shards_stitched': [os.path.dirname(f).rsplit('/', 1)[-1] for f in shard_files],
    'partition_check': 'exact (no dup/missing/unexpected)',
    'value_range_check': 'pass',
    'gold_crosscheck_max_abs_delta': float(delta.max()),
    'gold_macro_auc_recomputed': macro_here,
}
with open('verification_manifest.json', 'w') as f:
    json.dump(verification, f, indent=2)
print(json.dumps(verification, indent=2))
print('DONE -- pseudo_labels_qwen3_4b.csv is the Phase 4 label source', flush=True)
